In [1]:
import pandas as pd

In [ ]:
data = pd.read_csv("H1.csv")

In [6]:
print(data.shape)
print(data.head())

(40060, 31)
   IsCanceled  LeadTime  ArrivalDateYear ArrivalDateMonth  \
0           0       342             2015             July   
1           0       737             2015             July   
2           0         7             2015             July   
3           0        13             2015             July   
4           0        14             2015             July   

   ArrivalDateWeekNumber  ArrivalDateDayOfMonth  StaysInWeekendNights  \
0                     27                      1                     0   
1                     27                      1                     0   
2                     27                      1                     0   
3                     27                      1                     0   
4                     27                      1                     0   

   StaysInWeekNights  Adults  Children  ...      DepositType        Agent  \
0                  0       2         0  ...  No Deposit              NULL   
1                  0       2

In [7]:
data.isnull().sum()

IsCanceled                       0
LeadTime                         0
ArrivalDateYear                  0
ArrivalDateMonth                 0
ArrivalDateWeekNumber            0
ArrivalDateDayOfMonth            0
StaysInWeekendNights             0
StaysInWeekNights                0
Adults                           0
Children                         0
Babies                           0
Meal                             0
Country                        464
MarketSegment                    0
DistributionChannel              0
IsRepeatedGuest                  0
PreviousCancellations            0
PreviousBookingsNotCanceled      0
ReservedRoomType                 0
AssignedRoomType                 0
BookingChanges                   0
DepositType                      0
Agent                            0
Company                          0
DaysInWaitingList                0
CustomerType                     0
ADR                              0
RequiredCarParkingSpaces         0
TotalOfSpecialReques

In [8]:
data["TotalNights"] = (
    data["StaysInWeekendNights"] +
    data["StaysInWeekNights"]
)
data["AccommodationCost"] = (
    data["ADR"] * data["TotalNights"]
)
data["FoodCost"] = (
    data["Adults"] *
    data["TotalNights"] *
    500
)
data["TransportationCost"] = (
    data["Adults"] * 1000
)
data["TripCost"] = (
    data["AccommodationCost"] +
    data["FoodCost"] +
    data["TransportationCost"]
)

In [10]:
print(
    data[
        [
            "ADR",
            "TotalNights",
            "AccommodationCost",
            "FoodCost",
            "TransportationCost",
            "TripCost"
        ]
    ].head()
)

    ADR  TotalNights  AccommodationCost  FoodCost  TransportationCost  \
0   0.0            0                0.0         0                2000   
1   0.0            0                0.0         0                2000   
2  75.0            1               75.0       500                1000   
3  75.0            1               75.0       500                1000   
4  98.0            2              196.0      2000                2000   

   TripCost  
0    2000.0  
1    2000.0  
2    1575.0  
3    1575.0  
4    4196.0  


In [11]:
data[data["TotalNights"] == 0].shape

(384, 36)

In [12]:
data = data[data["TotalNights"] > 0]
print(data.shape)

(39676, 36)


In [13]:
features = [
    'LeadTime',
    'Adults',
    'Children',
    'Babies',
    'Meal',
    'MarketSegment',
    'CustomerType',
    'TotalNights',
    'IsRepeatedGuest',
    'BookingChanges'
]

X = data[features]

y = data["TripCost"]

print(X.head())
print(y.head())

   LeadTime  Adults  Children  Babies       Meal MarketSegment CustomerType  \
2         7       1         0       0  BB               Direct    Transient   
3        13       1         0       0  BB            Corporate    Transient   
4        14       2         0       0  BB            Online TA    Transient   
5        14       2         0       0  BB            Online TA    Transient   
6         0       2         0       0  BB               Direct    Transient   

   TotalNights  IsRepeatedGuest  BookingChanges  
2            1                0               0  
3            1                0               0  
4            2                0               0  
5            2                0               0  
6            2                0               0  
2    1575.0
3    1575.0
4    4196.0
5    4196.0
6    4214.0
Name: TripCost, dtype: float64


In [14]:
from sklearn.preprocessing import LabelEncoder
meal_encoder = LabelEncoder()
market_encoder = LabelEncoder()
customer_encoder = LabelEncoder()
X["Meal"] = meal_encoder.fit_transform(X["Meal"])

X["MarketSegment"] = market_encoder.fit_transform(X["MarketSegment"])

X["CustomerType"] = customer_encoder.fit_transform(X["CustomerType"])

C:\Users\Ajay Patel\AppData\Local\Temp\ipykernel_12012\2736196842.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Meal"] = meal_encoder.fit_transform(X["Meal"])
C:\Users\Ajay Patel\AppData\Local\Temp\ipykernel_12012\2736196842.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["MarketSegment"] = market_encoder.fit_transform(X["MarketSegment"])
C:\Users\Ajay Patel\AppData\Local\Temp\ipykernel_12012\2736196842.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Dat

In [15]:
print(X.head())

   LeadTime  Adults  Children  Babies  Meal  MarketSegment  CustomerType  \
2         7       1         0       0     0              2             2   
3        13       1         0       0     0              1             2   
4        14       2         0       0     0              5             2   
5        14       2         0       0     0              5             2   
6         0       2         0       0     0              2             2   

   TotalNights  IsRepeatedGuest  BookingChanges  
2            1                0               0  
3            1                0               0  
4            2                0               0  
5            2                0               0  
6            2                0               0  


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (31740, 10)
X_test : (7936, 10)
y_train: (31740,)
y_test : (7936,)


In [17]:
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

In [18]:
rf_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [19]:
y_pred = rf_model.predict(X_test)

In [20]:
print(y_pred[:10])

[ 9413.4228      6439.95546667  9423.34396336 10477.312
  6251.4374      3072.41409272  8299.7         1555.42792001
  7341.60927031  3091.45371496]


In [21]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import numpy as np

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("MSE :", mse)
print("RMSE:", rmse)
print("R2  :", r2)

MAE : 152.5163300152575
MSE : 113479.49598755084
RMSE: 336.86717855491776
R2  : 0.9939600637034385


In [22]:
import joblib

joblib.dump(rf_model, "cost_prediction_model.pkl")

['cost_prediction_model.pkl']